<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Check for correct environment path
import sys
from pathlib import Path

executable_path = Path(sys.executable)
active_env_idx = executable_path.parts.index("envs")
assert executable_path.parts[active_env_idx+1] == "default", "Wrong environment. Choose 'default' environment."
print("'Default' environment is loaded successfully!")

In [ ]:
from importlib.metadata import version

pkgs = ["matplotlib",  # Plotting library
        "numpy",       # PyTorch & TensorFlow dependency
        "tiktoken",    # Tokenizer
        "torch",       # Deep learning library
        "tensorflow",  # For OpenAI's pretrained weights
        "pandas"       # Dataset loading
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

---

# **Chapter 6: Finetuning for Text Classification**

---

## **Table of Contents**

---

## **Introduction**

In chapter 6, we transition from constructing a foundation model (coding, pretraining, and importing weights) to adapting it for practical, task-specific applications. Key Concepts and Objectives in this chapter are the following:

* **The Fine-tuning Shift**: This stage involves refining the model on smaller, labeled datasets to transform it from a general text-completer into a specialized "narrow expert".
* **Two Fine-Tuning Pathways**: 
    *   **Classification Fine-tuning**: Training the model to recognize predefined class labels, such as "spam" or "not spam".
    *   **Instruction Fine-tuning**: Improving the model’s ability to understand and execute complex tasks described in natural language prompts (the focus of Chapter 7).
* **The Practical Project**: The sources use a concrete case study: creating a spam detector for text messages. This involves modifying the pretrained architecture by replacing the original output layer (50,257 tokens) with a classification head containing only two nodes.
* **Chapter Roadmap**: The introduction outlines the necessary technical steps, including downloading and balancing the dataset, modifying the model, implementing evaluation utilities, and performing the final fine-tuning.

Ultimately, this section serves as the bridge between the foundational development of an LLM and its deployment as a functional, task-oriented AI tool.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/01.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.1</strong> The three main stages of coding an LLM. This chapter focus on stage 3 (step 8): fine-tuning a pretrained LLM as a classifier.
    </figcaption>
</figure>

---

### **6.1 Different categories of finetuning**

This section distinguishes between the two primary methods used to adapt a pretrained foundation model for specific applications.

The key categories and their characteristics include:

* **Instruction Fine-tuning**: This method trains the model to understand and execute tasks described in natural language prompts (e.g., "Summarize this text"). While highly versatile and capable of handling complex interactions, it requires larger datasets and significant computational resources.

<figure style="text-align: center; width: 500px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/02.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.2</strong></figcaption>
</figure>

* **Classification Fine-tuning**: This approach trains the model to recognize a predefined set of labels, such as "spam" or "not spam". It transforms the LLM into a highly specialized "narrow expert" that is confined to the classes encountered during training.

<figure style="text-align: center; width: 500px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/03.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.3</strong></figcaption>
</figure>

* **Key Trade-offs**: 
    * **Specialization vs. Generalization**: Classification models are easier to develop for specific tasks, whereas instruction-tuned models act as competent generalists.
    * **Efficiency**: Classification fine-tuning is ideal for projects with limited data or compute power, as it is less resource-intensive than instruction fine-tuning.

Ultimately, the choice between these methods depends on whether the goal is to create a flexible conversational assistant or a precise tool for categorizing data.

---

### **6.2 Preparing the dataset**

This section 2 details the initial phase of classification fine-tuning, focusing on acquiring and structuring data for the spam detection task.

Key technical and procedural steps include:

* **Dataset Acquisition**: The process begins by downloading and unzipping the SMS Spam Collection dataset, which is loaded into a pandas DataFrame for inspection.
* **Addressing Class Imbalance**: Initial analysis reveals a significant imbalance (4,825 "ham" vs. 747 "spam" messages). To ensure stable training and faster execution, the section implements undersampling to create a balanced dataset of 747 instances per class.
* **Label Encoding**: String labels are converted into numerical format, mapping "ham" to 0 and "spam" to 1, analogous to converting text into token IDs.
* **Data Partitioning**: The balanced data is shuffled and split into three subsets: 70% for training, 10% for validation, and 20% for testing.

Ultimately, this section establishes a clean, balanced foundation of labeled data, which is a prerequisite for modifying the model architecture and creating the specialized data loaders required for classification.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/04.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.4</strong> The three-stage process for classification fine-tuning an LLM. Stage 1 involves dataset preparation. Stage 2 focuses on model setup. Stage 3 covers fine-tuning and evaluating the model.
        </figcaption>
</figure>

First, we download and unzip the dataset:

In [ ]:
import requests
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    # Downloading the file
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    with open(zip_path, "wb") as out_file:
        # Pull data arrays in tiny 8KB block buffers to protect system memory layout
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                out_file.write(chunk)

    # Unzipping the file
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Add .tsv file extension
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")


try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (requests.exceptions.RequestException, TimeoutError) as e:
    print(f"Primary URL failed: {e}. Trying backup URL...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)



# The book originally used the following code below
# However, urllib uses older protocol settings that
# can cause problems for some readers using a VPN.
# The `requests` version above is more robust
# in that regard.

"""
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    # Downloading the file
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # Unzipping the file
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Add .tsv file extension
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as e:
    print(f"Primary URL failed: {e}. Trying backup URL...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
"""

The dataset is saved as a tab-separated text file, which we can load into a pandas DataFrame

In [ ]:
import pandas as pd

df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
df

When we check the class distribution, we see that the data contains "ham" (i.e., "not spam") much more frequently than "spam"

In [ ]:
print(df["Label"].value_counts())

For simplicity, and because we prefer a small dataset for educational purposes anyway (it will make it possible to finetune the LLM faster), we subsample (undersample) the dataset so that it contains 747 instances from each class.

(Next to undersampling, there are several other ways to deal with class balances, but they are out of the scope of a book on LLMs; you can find examples and more information in the ["imbalanced-learn" user guide](https://imbalanced-learn.org/stable/user_guide.html))

In [ ]:
def create_balanced_dataset(df):
    
    # Count the instances of "spam"
    num_spam = df[df["Label"] == "spam"].shape[0]
    
    # Randomly sample "ham" instances to match the number of "spam" instances
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    
    # Combine ham "subset" with "spam"
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])

    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Next, we change the string class labels "ham" and "spam" into integer class labels 0 and 1:

In [ ]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})    

In [ ]:
balanced_df

Let's now define a function that randomly divides the dataset into training, validation, and test subsets

In [ ]:
def random_split(df, train_frac, validation_frac):
    # Shuffle the entire DataFrame
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # Calculate split indices
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # Split the DataFrame
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
# Test size is implied to be 0.2 as the remainder

train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

If you have worked in machine learning area, you are already familiar with `train_test_split` function from Scikit-Learn library that is commonly use to create train/val/test sets:

```python
from sklearn.model_selection import train_test_split

train_df, val_test_df = train_test_split(balanced_df, test_size=0.3, random_state=123, shuffle=True)

validation_df, test_df = train_test_split(val_test_df, test_size=0.67, random_state=123, shuffle=True)

```


---

### **6.3 Creating data loaders**

This section details the process of structuring the balanced spam dataset into PyTorch-compatible batches for fine-tuning.

Key technical and procedural steps include:

* **Handling Variable Lengths**: Unlike the uniform text chunks used during pretraining, SMS messages vary in length. To create efficient batches, the source opts for padding all messages to the length of the longest one in the batch rather than truncating them, which avoids information loss.
* **The Padding Mechanism**: The model uses the `<|endoftext|>` token (ID 50256) as a padding character. Shorter messages are extended with this ID until they match the target length, ensuring uniform tensor shapes within each batch.
* **`SpamDataset` Implementation**: A custom PyTorch `Dataset` class is developed to automate the pre-tokenization of messages and handle the specific padding or truncation logic required for the classification task.
* **Target Labels**: A major shift from pretraining is that the data loaders now return numerical class labels (0 for "not spam" and 1 for "spam") as the target, rather than a shifted sequence of next-token IDs.
* **Loader Configuration**: The datasets are plugged into standard PyTorch `DataLoaders` with a specified batch size (e.g., 8). Shuffling is enabled for the training loader to prevent the model from learning the order of samples.

Ultimately, this section completes the data preparation phase by providing a robust pipeline that delivers batches of tokenized, padded text and their corresponding labels to the model during the fine-tuning process.


Note that the text messages have different lengths; if we want to combine multiple training examples in a batch, we have to either
1. truncate all messages to the length of the shortest message in the dataset or batch
2. pad all messages to the length of the longest message in the dataset or batch

We choose option 2 and pad all messages to the longest message in the dataset. For that, we use `<|endoftext|>` as a padding token, as discussed in chapter 2:

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/06.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.6</strong>  The input text preparation process. First, each input text message is converted into a sequence of token IDs. Then, to ensure uniform sequence lengths, shorter sequences are padded with a padding token (in this case, token ID 50256) to match the length of the longest sequence.
        </figcaption>
</figure>

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

The `SpamDataset` class below identifies the longest sequence in the training dataset and adds the padding token to the others to match that sequence length

In [ ]:
import torch
from torch.utils.data import Dataset


class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        # Pre-tokenize texts
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # Truncate sequences if they are longer than max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # Pad sequences to the longest sequence
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length
        # Note: A more pythonic version to implement this method
        # is the following, which is also used in the next chapter:
        # return max(len(encoded_text) for encoded_text in self.encoded_texts)

        # A more abstract version to implement this method is the following
        # return max(map(len, self.encoded_texts))

In [ ]:
train_dataset = SpamDataset(
    csv_file="train.csv",
    max_length=None,
    tokenizer=tokenizer
)

print(train_dataset.max_length)

We also pad the validation and test set to the longest training sequence. Note that validation and test set samples that are longer than the longest training example are being truncated via `encoded_text[:self.max_length]` in the `SpamDataset` code. This behavior is entirely optional, and it would also work well if we set `max_length=None` in both the validation and test set cases.

In [ ]:
val_dataset = SpamDataset(
    csv_file="validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset = SpamDataset(
    csv_file="test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

Next, we use the dataset to instantiate the data loaders, which is similar to creating the data loaders in previous chapters:

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/07.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.7</strong> A single training batch consisting of eight text messages represented as token IDs. Each text message consists of 120 token IDs. A class label array stores the eight class labels corresponding to the text messages, which can be either 0 (“not spam”) or 1 (“spam”).
        </figcaption>
</figure>

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

As a verification step, we iterate through the data loaders and ensure that the batches contain 8 training examples each, where each training example consists of 120 tokens

In [ ]:
print("Train loader:")
for input_batch, target_batch in train_loader:
    pass

print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

Lastly, let's print the total number of batches in each dataset:

In [ ]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

---

### **6.4 Initializing a model with pretrained weights**

This section details the preparation of the base LLM for the subsequent classification fine-tuning process.

Key technical and procedural steps include:

* **Model Configuration**: The `GPTModel` is initialized using the GPT-2 small (124M) settings. Critical updates are made to match OpenAI's specific training parameters, including increasing the context length to 1,024 tokens and enabling bias vectors (`qkv_bias=True`) in the attention layers.
* **Weight Integration**: Pretrained weights are downloaded from OpenAI and mapped into the local architecture using the `load_weights_into_gpt` function.
* **Functional Validation**: To ensure the weights were mapped correctly, the model is tested with a standard prompt; producing coherent text confirms that the architecture and weights are properly aligned.
* **Performance Baseline**: A preliminary test prompts the model to classify a message as "spam" or "not spam." The output shows that the model struggles to follow instructions at this stage, as it has only undergone pretraining and lacks the task-specific fine-tuning required for classification.

Ultimately, this section establishes the pretrained foundation model as a starting point before it is structurally modified for its new role as a classifier.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/08.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.8</strong> The three-stage process for classification fine-tuning the LLM. Having completed stage 1, preparing the dataset, we now must initialize the LLM, which we will then fine-tune to classify spam messages.
        </figcaption>
</figure>

In [ ]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"Dataset length {train_dataset.max_length} exceeds model's context "
    f"length {BASE_CONFIG['context_length']}. Reinitialize data sets with "
    f"`max_length={BASE_CONFIG['context_length']}`"
)

In [ ]:
from llms_from_scratch.ch06.main_chapter_code.gpt_download import download_and_load_gpt2
from llms_from_scratch.ch06.main_chapter_code.previous_chapters import GPTModel, load_weights_into_gpt

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

To ensure that the model was loaded correctly, let's double-check that it generates coherent text:

In [ ]:
from llms_from_scratch.ch06.main_chapter_code.previous_chapters import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)

text_1 = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

Before we finetune the model as a classifier, let's see if the model can perhaps already classify spam messages via prompting:

In [ ]:
text_2 = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_2, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

As we can see, the model is not very good at following instructions. This is expected, since it has only been pretrained and not instruction-finetuned (instruction finetuning will be covered in the next chapter).

#### **Exercise 6.1 Increasing the context length**

Pad the inputs to the maximum number of tokens the model supports and observe how it affects the predictive performance.

#### **Solution 6.1**

---

### **6.5 Adding a classification head**

This section details the structural modifications required to transform a pretrained generative GPT model into a specialized text classifier.

Key technical and procedural steps include:

* **Replacing the Output Layer**: The model's original output head, which mapped 768 hidden units to 50,257 vocabulary tokens, is replaced with a smaller classification head. For this project, the new layer projects the hidden units to just two output nodes representing "spam" and "not spam".
* **Selective Fine-tuning**: To ensure efficient training, all model parameters are initially frozen (set to `requires_grad = False`). While training only the new head is possible, performance is significantly improved by also unfreezing the final LayerNorm and the last transformer block, allowing these layers to adapt to task-specific linguistic patterns.
* **Last-Token Logic**: During classification, the model only uses the output vector corresponding to the last input token rather than averaging all tokens. Because GPT models use a causal attention mask, information flows strictly from left to right. Consequently, the final token is the only one that has "attended to" every preceding word in the sequence, making its vector a comprehensive summaryof the entire input message.

Ultimately, these modifications repurpose the pretrained "foundation" model, allowing it to interpret the context of a full message and assign a single definitive class label.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/09.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.9</strong> Adapting a GPT model for spam classification by altering its architecture. Initially, the model’s linear output layer mapped 768 hidden units to a vocabulary of 50,257 tokens. To detect spam, we replace this layer with a new output layer that maps the same 768 hidden units to just two classes, representing “spam” and “not spam.”
        </figcaption>
</figure>

In this section, we are modifying the pretrained LLM to make it ready for classification finetuning. Let's take a look at the model architecture first:

In [ ]:
print(model)

Above, we can see the architecture we implemented in chapter 4 neatly laid out. The goal is to replace and finetune the output layer. To achieve this, we first freeze the model, meaning that we make all layers non-trainable:

In [ ]:
for param in model.parameters():
    param.requires_grad = False

Then, we replace the output layer (`model.out_head`), which originally maps the layer inputs to 50,257 dimensions (the size of the vocabulary), with binary classification head (predicting 2 classes, "spam" and "not spam"). The new head will be trainable by default. Note that we use `BASE_CONFIG["emb_dim"]` (which is equal to 768 in the `"gpt2-small (124M)"` model) to keep the code below more general.

In [ ]:
torch.manual_seed(123)

num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

Technically, it's sufficient to only train the output layer. However, as I found in [Finetuning Large Language Models](https://magazine.sebastianraschka.com/p/finetuning-large-language-models), experiments show that finetuning additional layers can noticeably improve the performance. So, we are also making the last transformer block and the final `LayerNorm` module connecting the last transformer block to the output layer trainable.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/10.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.10</strong> The GPT model includes 12 repeated transformer blocks. Alongside the output layer, we set the final LayerNorm and the last transformer block as trainable. The remaining 11 transformer blocks and the embedding layers are kept nontrainable.
        </figcaption>
</figure>

In [ ]:
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

We can still use this model similar to before in previous chapters. For example, let's feed it some text input:

In [ ]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape) # shape: (batch_size, num_tokens)

What's different compared to previous chapters is that it now has two output dimensions instead of 50,257:

In [ ]:
with torch.no_grad():
    outputs = model(inputs)

print("Outputs:\n", outputs)
print("Outputs dimensions:", outputs.shape) # shape: (batch_size, num_tokens, num_classes)

As discussed in previous chapters, for each input token, there's one output vector. Since we fed the model a text sample with 4 input tokens, the output consists of 4 2-dimensional output vectors above.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/11.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.11</strong> The GPT model with a four-token example input and output. The output tensor consists of two columns due to the modified output layer. We are only interested in the last row corresponding to the last token when fine-tuning the model for spam classification.
        </figcaption>
</figure>

>**Important note**: In chapter 3, we discussed the attention mechanism, which connects each input token to each other input token. We then introduced the causal attention mask that is used in GPT-like models; this causal mask lets a current token only attend to the current and previous token positions. Based on this causal attention mechanism, the 4th (last) token contains the most information among all tokens because it's the only token that includes information about all other tokens. Hence, we are particularly interested in this last token, which we will finetune for the spam classification task.

In [ ]:
print("Last output token:", outputs[:, -1, :])

<figure style="text-align: center; width: 500px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/12.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.12</strong> The causal attention mechanism, where the attention scores between input tokens are displayed in a matrix format. The empty cells indicate masked positions due to the causal attention mask, preventing tokens from attending to future tokens. The values in the cells represent attention scores; the last token, time, is the only one that computes attention scores for all preceding tokens.
        </figcaption>
</figure>

#### **Exercise 6.2 Fine-tuning the whole model**
Instead of fine-tuning just the final transformer block, fine-tune the entire model and assess the effect on predictive performance.

#### **Solution 6.2**

#### **Exercise 6.3 Fine-tuning the first vs. last token**
Try fine-tuning the first output token. Notice the changes in predictive performance compared to fine-tuning the last output token.

#### **Solution 6.3**

---

### **6.6 Calculating the classification loss and accuracy**

This section details the implementation of numeric evaluation utilities required to monitor the model's progress during the fine-tuning phase.

Key technical and conceptual points include:

* **Label Prediction Logic**: To convert model outputs into class labels ("spam" or "not spam"), the system focuses on the vector corresponding to the last input token. While the softmax function can be used to generate probabilities, applying `torch.argmax` directly to the raw logits is sufficient to identify the predicted class index (0 or 1).
* **Accuracy Metric**: The source introduces the `calc_accuracy_loader` function, which iterates through a dataset to calculate the proportion of correct predictions.
* **Loss as a Proxy**: Because classification accuracy is not a differentiable function, the model is optimized using cross-entropy loss. A critical adjustment is made to the `calc_loss_batch` utility to ensure it only calculates loss based on the final token's output rather than the entire sequence.
* **Establishing a Baseline**: Before training begins, initial loss and accuracy are calculated across the training, validation, and test sets. These values (e.g., a training loss of ~2.453) serve as the numerical baseline that the fine-tuning process aims to improve.

Ultimately, this section provides the mathematical and programmatic framework to objectively measure how well the repurposed LLM is learning its new task as a specialized classifier.

<figure style="text-align: center; width: 500px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/13.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.13</strong> The three-stage process for classification fine-tuning the LLM. We've completed the first six steps. We are now ready to undertake the last step of stage 2: implementing the functions to evaluate the model’s performance to classify spam messages before, during, and after the fine-tuning.
        </figcaption>
</figure>

Before explaining the loss calculation, let's have a brief look at how the model outputs are turned into class labels.

<figure style="text-align: center; width: 600px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/14.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.14</strong> The model outputs corresponding to the last token are converted into probability scores for each input text. The class labels are obtained by looking up the index position of the highest probability score. The model predicts the spam labels incorrectly because it has not yet been trained.
        </figcaption>
</figure>

In [ ]:
print("Last output token:", outputs[:, -1, :])

Similar to chapter 5, we convert the outputs (logits) into probability scores via the `softmax` function and then obtain the index position of the largest probability value via the `argmax` function:

In [ ]:
probas = torch.softmax(outputs[:, -1, :], dim=-1)
label = torch.argmax(probas)
print("Class label:", label.item())

Note that the softmax function is optional here, as explained in chapter 5, because the largest outputs correspond to the largest probability scores

In [ ]:
logits = outputs[:, -1, :]
label = torch.argmax(logits)
print("Class label:", label.item())

We can apply this concept to calculate the so-called classification accuracy, which computes the percentage of correct predictions in a given dataset. To calculate the classification accuracy, we can apply the preceding `argmax`-based prediction code to all examples in a dataset and calculate the fraction of correct predictions as follows:

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # Logits of last output token
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples

Let's apply the function to calculate the classification accuracies for the different datasets:

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes

torch.manual_seed(123) # For reproducibility due to the shuffling in the training data loader

train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

As we can see, the prediction accuracies are not very good, since we haven't finetuned the model, yet.

Before we can start finetuning (/training), we first have to define the loss function we want to optimize during training. The goal is to maximize the spam classification accuracy of the model; however, classification accuracy is not a differentiable function. Hence, instead, we minimize the cross-entropy loss as a proxy for maximizing the classification accuracy (you can learn more about this topic in lecture 8 of my freely available [Introduction to Deep Learning](https://sebastianraschka.com/blog/2021/dl-course.html#l08-multinomial-logistic-regression--softmax-regression) class)

The `calc_loss_batch` function is the same here as in chapter 5, except that we are only interested in optimizing the last token `model(input_batch)[:, -1, :]` instead of all tokens `model(input_batch)`.

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]  # Logits of last output token
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

The `calc_loss_loader` is exactly the same as in chapter 5:

In [ ]:
# Same as in chapter 5
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

Using the `calc_closs_loader`, we compute the initial training, validation, and test set losses before we start training:

In [ ]:
with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

In the next section, we train the model to improve the loss values and consequently the classification accuracy.

---

### **6.7 Finetuning the model on supervised data**

This section details the implementation and execution of the training loop required to transform the pretrained LLM into a high-performing spam classifier.

Key technical and procedural points include:

* **The Training Function**: The source introduces `train_classifier_simple`, a function that closely mirrors the pretraining loop from Chapter 5. The primary differences are that it tracks the number of training examples seen (rather than tokens) and calculates classification accuracy after each epoch instead of generating text samples.
* **The Core Loop**: The process follows standard PyTorch steps: iterating over epochs and batches, resetting gradients, calculating loss (specifically targeting the last token's logits), performing backpropagation, and updating model weights.
* **Performance and Convergence**: In a practical test of five epochs, the model's training loss dropped from approximately 2.15 to 0.08, while training accuracy reached 100% and validation accuracy hit 97.5%.
* **Visualizing Progress**: The section utilizes Matplotlib to plot loss and accuracy curves. The sharp downward slope of the loss and the plateauing of accuracy indicate that the model learned effectively from the training data while generalizing well to unseen validation data with minimal overfitting.
* **Final Validation**: After the training loop completes, a comprehensive evaluation is performed across the entire dataset (training, validation, and test sets) to confirm the model's final predictive performance.

Ultimately, this section completes the technical transition, demonstrating that a relatively small amount of fine-tuning can repurpose a general foundation model into a highly accurate, specialized tool.

<figure style="text-align: center; width: 600px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/15.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.15</strong> A typical training loop for training deep neural networks in PyTorch consists of several steps, iterating over the batches in the training set for several epochs. In each loop, we calculate the loss for each training set batch to determine loss gradients, which we use to update the model weights to minimize the training set loss.
        </figcaption>
</figure>

In [ ]:
# Overall the same as `train_model_simple` in chapter 5
def train_classifier_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                            eval_freq, eval_iter):
    # Initialize lists to track losses and examples seen
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            examples_seen += input_batch.shape[0] # New: track examples instead of tokens
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Calculate accuracy after each epoch
        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

The `evaluate_model` function used in the `train_classifier_simple` is the same as the one we used in chapter 5:

In [ ]:
# Same as chapter 5
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

The training takes about 5 minutes on a M3 MacBook Air laptop computer and less than half a minute on a V100 or A100 GPU

In [ ]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Similar to chapter 5, we use matplotlib to plot the loss function for the training and validation set:

In [ ]:
import matplotlib.pyplot as plt

def plot_values(epochs_seen, examples_seen, train_values, val_values, label="loss"):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_values, label=f"Training {label}")
    ax1.plot(epochs_seen, val_values, linestyle="-.", label=f"Validation {label}")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    # Create a second x-axis for examples seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(examples_seen, train_values, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Examples seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig(f"{label}-plot.pdf")
    plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses)

Above, based on the downward slope, we see that the model learns well. Furthermore, the fact that the training and validation loss are very close indicates that the model does not tend to overfit the training data.

 Similarly, we can plot the accuracy below:

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_accs))

plot_values(epochs_tensor, examples_seen_tensor, train_accs, val_accs, label="accuracy")

Based on the accuracy plot above, we can see that the model achieves a relatively high training and validation accuracy after epochs 4 and 5. However, we have to keep in mind that we specified `eval_iter=5` in the training function earlier, which means that we only estimated the training and validation set performances. We can compute the training, validation, and test set performances over the complete dataset as follows below

In [ ]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

We can see that the training and validation set performances are practically identical. However, based on the slightly lower test set performance, we can see that the model overfits the training data to a very small degree, as well as the validation data that has been used for tweaking some of the hyperparameters, such as the learning rate. This is normal, however, and this gap could potentially be further reduced by increasing the model's dropout rate (`drop_rate`) or the `weight_decay` in the optimizer setting.

---

### **6.8 Using the LLM as a spam classifier**

This section describes the final implementation phase where the fine-tuned model is deployed to categorize new, unseen text messages.

Key technical and procedural points include:

* **The `classify_review` Function**: A dedicated utility is introduced to streamline the classification process. It replicates the preprocessing steps of the `SpamDataset`, including tokenization and padding/truncation to match the training context length.
* **Prediction Logic**: During inference, the function extracts the logits from the final input token to determine the class label.
* **Human-Readable Output**: The function converts the model's numerical predictions (0 or 1) into descriptive strings, returning "spam" or "not spam".
* **Model Persistence**: The section details saving the fine-tuned model's `state_dict` (e.g., as `review_classifier.pth`). This allows the trained weights to be loaded in future sessions, eliminating the need to repeat the expensive fine-tuning process.

Ultimately, this section demonstrates the practical application of the repurposed foundation model as a **functional, task-oriented AI tool**.

<figure style="text-align: center; width: 750px; margin: 0 auto;">
    <img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch06_compressed/18.webp" width=100%>
    <figcaption style="text-align: left; margin-top: 8px;">
        <strong>Figure 6.18</strong> The three-stage process for classification fine-tuning our LLM. Step 10 is the final step of stage 3—using the fine-tuned model to classify new spam messages.
        </figcaption>
</figure>

Finally, let's use the finetuned GPT model in action. The `classify_review` function below implements the data preprocessing steps similar to the `SpamDataset` we implemented earlier. Then, the function returns the predicted integer class label from the model and returns the corresponding class name:

In [ ]:
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    model.eval()

    # Prepare inputs to the model
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]
    
    # Truncate sequences if they too long
    max_len = min(max_length,supported_context_length) if max_length else supported_context_length
    input_ids = input_ids[:max_len]
    assert max_length is not None, (
        "max_length must be specified. If you want to use the full model context, "
        "pass max_length=model.pos_emb.weight.shape[0]."
    )
    assert max_length <= supported_context_length, (
        f"max_length ({max_length}) exceeds model's supported context length ({supported_context_length})."
    )    
        
    # Pad sequences to the longest sequence
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0) # add batch dimension

    # Model inference
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]  # Logits of the last output token
    predicted_label = torch.argmax(logits, dim=-1).item()

    # Return the classified result
    return "spam" if predicted_label == 1 else "not spam"

Let's try it out on a few examples below:

In [ ]:
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)

print(classify_review(
    text_1, model, tokenizer, device, max_length=train_dataset.max_length
))

In [ ]:
text_2 = (
    "Hey, just wanted to check if we're still on"
    " for dinner tonight? Let me know!"
)

print(classify_review(
    text_2, model, tokenizer, device, max_length=train_dataset.max_length
))

Finally, let's save the model in case we want to reuse the model later without having to train it again:

In [ ]:
torch.save(model.state_dict(), "review_classifier.pth")

Then, in a new session, we could load the model as follows:

In [ ]:
model_state_dict = torch.load("review_classifier.pth", map_location=device, weights_only=True)
model.load_state_dict(model_state_dict)

---

## **Summary and takeaways**

* There are different strategies for fine-tuning LLMs, including classification fine-tuning and instruction fine-tuning.
* Classification fine-tuning involves replacing the output layer of an LLM via a small classification layer.
* In the case of classifying text messages as “spam” or “not spam,” the new classification layer consists of only two output nodes. Previously, we used the number
of output nodes equal to the number of unique tokens in the vocabulary (i.e., 50,256).
* Instead of predicting the next token in the text as in pretraining, classification fine-tuning trains the model to output a correct class label—for example,
“spam” or “not spam.”
* The model input for fine-tuning is text converted into token IDs, similar to pretraining.
* Before fine-tuning an LLM, we load the pretrained model as a base model.
* Evaluating a classification model involves calculating the classification accuracy (the fraction or percentage of correct predictions).
* Fine-tuning a classification model uses the same cross entropy loss function as when pretraining the LLM.

---

## **Summplementary Materials**

* See the [gpt_class_finetune.py](../../../src/llms_from_scratch/ch06/main_chapter_code/gpt_class_finetune.py) script, a self-contained script for classification finetuning.
* In addition, interested readers can find an introduction to parameter-efficient training with low-rank adaptation (LoRA) in [appendix E](../../appendix-E/01_main-chapter-code/appendix-E.ipynb).